# Day 2: Grover, four candidates, and one marked result

Material development: **Gage Plott**.

Follow the complete example, then change only the two indicated values with
the presenter. Use a working Python (PHYS 3810 Guided) kernel and run from top to bottom.

The candidates are `00`, `01`, `10`, and `11`. The rightmost bit is **qubit 0**:
`01` has qubit 0 set; `10` has qubit 1 set. A label is text, so keep its quotes.

The teaching oracle is constructed for the marked label we choose. It changes
that label's amplitude sign; the diffusion step then changes the amplitudes.
One oracle-plus-diffusion pair is one **Grover iteration**. We import these
operations from the tested course library rather than rebuilding them here.
The example does not search an external database or demonstrate quantum advantage.

## 1. Run the support cell unchanged

It locates the course files and imports the circuit and plotting tools. You do
not need to write the path-search or checking code. Keep the complete extracted
`runtime` folder together. Each run uses a local simulator and seed 2026.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
from importlib.metadata import version

# Locate only this extracted runtime/repository tree, never a fixed developer path.
repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "phys3810" / "grover.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Keep the runtime folder together and launch Jupyter from its root.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert version("qiskit") == "2.6.1", "Select the pinned classroom kernel."

from math import isclose, sqrt  # noqa: E402 - after standalone path/environment setup

import matplotlib.pyplot as plt  # noqa: E402 - after standalone path/environment setup
from IPython.display import display  # noqa: E402 - after standalone path/environment setup
from qiskit.primitives import StatevectorSampler  # noqa: E402 - after standalone path/environment setup
from qiskit.quantum_info import Statevector  # noqa: E402 - after standalone path/environment setup
from qiskit.visualization import plot_histogram  # noqa: E402 - after standalone path/environment setup
from src.phys3810.grover import grover_circuit  # noqa: E402 - after standalone path/environment setup

print("Course source:", Path(sys.modules[grover_circuit.__module__].__file__).resolve())


## 2. Choose the two guided values

**Editable:** `target` may be `"00"`, `"01"`, `"10"`, or `"11"`.
`iterations` may be the whole number **0, 1, or 2**. Keep the other settings as
written. Change one value at a time, then rerun every cell below this one.

With four candidates and one marked result, the exact marked probabilities are
$1/4$, $1$, and $1/4$ after zero, one, and two iterations respectively. These
values follow $\sin^2((2r+1)\pi/6)$ for iteration count $r$. The curve is not
monotonically increasing: applying more iterations can undo amplification.

**Predict before running:** for the selected iteration count, will the marked
label occur in approximately a quarter of the samples, or in all of them?


In [ ]:
target = "01"
iterations = 1
shots = 512
seed = 2026

assert target in ("00", "01", "10", "11"), "Choose one of the four two-bit labels."
assert type(iterations) is int and iterations in (0, 1, 2), "Choose 0, 1, or 2 iterations."


## 3. Build a fresh circuit and read the exact probability

`grover_circuit(2, [target], iterations)` calls the course function with two
qubits, a one-item list containing the marked label, and the chosen number of
iterations. It starts by preparing all four possibilities equally.

The simulator can also calculate exact probabilities before sampling. The
read-only checks compare the circuit with the three analytic values above.
`int(target, 2)` interprets the label as a binary integer; for example, `01`
is index 1 and `10` is index 2. This is an index, **not a phase estimate**.


In [ ]:
search = grover_circuit(2, [target], iterations)
probabilities = Statevector.from_instruction(search).probabilities()
target_probability = float(probabilities[int(target, 2)])
expected_probability = [1 / 4, 1, 1 / 4][iterations]
assert isclose(target_probability, expected_probability, rel_tol=0, abs_tol=1e-12)
assert search.num_clbits == 0
print(f"Target {target}; iterations={iterations}; exact probability={target_probability:.6f}")
print("Independent expected probability:", expected_probability)


The diagram below uses the current values. It is rebuilt from the beginning,
so rerunning the build cell cannot accumulate extra iterations.


In [ ]:
circuit_title = f"Target {target}; {iterations} Grover iterations"
figure = search.draw("mpl", fold=20)
figure.suptitle(circuit_title)
display(figure)
plt.close(figure)


## 4. Sample and compare

The measured copy leaves `search` unchanged. Counts and the caption are derived
from the current parameters. At one iteration, the ideal marked outcome is
certain. At zero or two iterations, finite samples fluctuate around probability
one quarter; the software check allows six standard deviations of fluctuation.


In [ ]:
measured = search.measure_all(inplace=False)
counts = StatevectorSampler(seed=seed).run([measured], shots=shots).result()[0].data.meas.get_counts()
observed_probability = counts.get(target, 0) / shots
expected_count = shots * expected_probability
sigma = sqrt(shots * expected_probability * (1 - expected_probability))
assert sum(counts.values()) == shots
assert set(counts) <= {"00", "01", "10", "11"}
assert abs(counts.get(target, 0) - expected_count) <= 6 * sigma + 1e-9
assert search.num_clbits == 0
print("Counts:", counts)
print(f"Observed fraction for {target}: {observed_probability:.6f}")
histogram_title = (
    f"Target {target}; iterations={iterations}; exact P={expected_probability:g}; {shots} samples"
)
figure = plot_histogram(counts, figsize=(8, 3.5), title=histogram_title)
display(figure)
plt.close(figure)


## 5. Compare zero, one, and two iterations

This read-only display makes three independent circuit calls using the same
current target. It does not add gates to the earlier circuit. The black dots
are the independent analytic probabilities; the bars are exact simulated
probabilities. The sample histogram above is a different, finite-shot view.


In [ ]:
comparison_probabilities = [
    float(Statevector.from_instruction(grover_circuit(2, [target], 0)).probabilities()[int(target, 2)]),
    float(Statevector.from_instruction(grover_circuit(2, [target], 1)).probabilities()[int(target, 2)]),
    float(Statevector.from_instruction(grover_circuit(2, [target], 2)).probabilities()[int(target, 2)]),
]
assert isclose(comparison_probabilities[0], 1 / 4, rel_tol=0, abs_tol=1e-12)
assert isclose(comparison_probabilities[1], 1, rel_tol=0, abs_tol=1e-12)
assert isclose(comparison_probabilities[2], 1 / 4, rel_tol=0, abs_tol=1e-12)
comparison_title = f"Target {target}: more iterations need not mean more success"
figure, axes = plt.subplots(figsize=(8, 3.5))
axes.bar([0, 1, 2], comparison_probabilities, label="Exact circuit probability")
axes.plot([0, 1, 2], [1 / 4, 1, 1 / 4], "ko", label="Independent prediction")
axes.set(xticks=[0, 1, 2], xlabel="Grover iterations", ylabel=f"Probability of {target}",
         ylim=(0, 1.15), title=comparison_title)
axes.legend()
figure.tight_layout()
display(figure)
plt.close(figure)
print(f"Target {target}; probabilities for 0, 1, 2 iterations:", comparison_probabilities)


## 6. Change, rerun, and discuss

Together, change `target` from `"01"` to `"10"` and rerun all cells below the
parameter cell. The favored label moves, but the expected probabilities do not.
Then change `iterations` from 1 to 2, rerun below, and compare with the earlier
prediction. Zero iterations is a useful starting-state comparison.

If time is short, stop after the one-iteration histogram and show the comparison
plot as a prepared demonstration. Do not rush through additional algorithms.
If a variable is missing or results look stale, restart the kernel and run all
cells in order. Save your working copy when finished.

The course function supports more inputs, but this guided route deliberately
uses only two qubits, one marked label, and 0/1/2 iterations. Oracle construction,
simulation cost, and reading results all have costs; this tiny demonstration
is not evidence of scalable quantum speedup.
